<a href="https://colab.research.google.com/github/adonaytetemke-wq/cofolding-benchmark/blob/main/insulin_folding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Insulin: How a Protein Folds into Its Working Shape
Insulin is a small protein hormone used as a remedy for diabetes. It is made of two chains (A and B) that fold and lock together with disulfide bonds. Here we load its real experimental structure, color its helices, highlight its disulfide bonds, and compare it with an AlphaFold prediction.

## 1. Setup

In [1]:
!pip install py3Dmol requests -q

In [2]:
import requests, py3Dmol

def fetch(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text

## 2. Load the experimental structure (PDB: 4INS)
The Protein Data Bank stores structures solved by X-ray crystallography.

In [3]:
pdb_text = fetch("https://files.rcsb.org/download/4INS.pdb")

chains = {}
for line in pdb_text.splitlines():
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        chains.setdefault(line[21], 0)
        chains[line[21]] += 1
print("Chains and residue counts:", chains)

Chains and residue counts: {'A': 21, 'B': 30, 'C': 21, 'D': 30}


## 3. Secondary structure and disulfide bonds

In [4]:
helix = [l for l in pdb_text.splitlines() if l.startswith("HELIX")]
sheet = [l for l in pdb_text.splitlines() if l.startswith("SHEET")]
ssbond = [l for l in pdb_text.splitlines() if l.startswith("SSBOND")]
print("Helices:", len(helix))
print("Sheet strands:", len(sheet))
print("Disulfide bonds:", len(ssbond))

Helices: 6
Sheet strands: 2
Disulfide bonds: 6


## 4. Interactive 3D view
Helices, sheets and loops are colored by type; yellow sticks are the disulfide bonds that hold the chains together.

In [5]:
view = py3Dmol.view(width=500, height=450)
view.addModel(pdb_text, "pdb")
view.setStyle({"cartoon": {"colorscheme": "ssJmol"}})
view.addStyle({"resn": "CYS"}, {"stick": {"colorscheme": "yellowCarbon"}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. Color by chain

In [6]:
view = py3Dmol.view(width=500, height=450)
view.addModel(pdb_text, "pdb")
view.setStyle({"cartoon": {"colorscheme": "chain"}})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
!pip install -q py3Dmol requests

In [8]:
import requests
pdb_text = requests.get("https://files.rcsb.org/download/4INS.pdb", timeout=30).text
print("4INS downloaded:", len(pdb_text), "characters")

4INS downloaded: 144585 characters


## 6. Compare with AlphaFold
AlphaFold predicts structure from the amino acid sequence alone. Its insulin entry (UniProt P01308) is the full precursor, preproinsulin, so it includes extra pieces that are cut away in mature insulin.

In [9]:
import requests, py3Dmol

def fetch(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text

def get_alphafold(uniprot="P01308"):
    try:
        r = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}", timeout=30)
        r.raise_for_status()
        return fetch(r.json()[0]["pdbUrl"])
    except Exception as e:
        print("API lookup failed:", e)
    for v in (6, 5, 4):
        try:
            return fetch(f"https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-model_v{v}.pdb")
        except Exception:
            pass
    raise RuntimeError("Could not download the AlphaFold model. Try again in a few minutes.")

af_text = get_alphafold()
print("AlphaFold model downloaded:", len(af_text), "characters")

# Experimental structure (4INS)
v1 = py3Dmol.view(width=450, height=380)
v1.addModel(pdb_text, "pdb")
v1.setStyle({"cartoon": {"colorscheme": "chain"}})
v1.zoomTo()
print("Experimental structure (4INS)")
v1.show()

# AlphaFold prediction, colored by confidence (blue = high, red = low)
v2 = py3Dmol.view(width=450, height=380)
v2.addModel(af_text, "pdb")
v2.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
v2.zoomTo()
print("AlphaFold prediction (UniProt P01308)")
v2.show()

AlphaFold model downloaded: 72575 characters
Experimental structure (4INS)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

AlphaFold prediction (UniProt P01308)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

- **Left:** experimental structure (4INS)
- **Right:** AlphaFold prediction (rainbow from start to end)

The helical core matches well. Differences come from the precursor's extra segments and from experiments showing a stable, assembled form.

In [10]:
!pip install -q biopython

import io
from Bio.PDB import PDBParser, Superimposer

parser = PDBParser(QUIET=True)
exp_struct = parser.get_structure("4INS", io.StringIO(pdb_text))[0]
af_struct = parser.get_structure("AF", io.StringIO(af_text))[0]["A"]

# In human preproinsulin (UniProt P01308):
# Signal peptide: 1-24 | B-chain: 25-54 (30 aa) | C-peptide: 57-87 | A-chain: 90-110 (21 aa)
exp_ca = [res["CA"] for res in exp_struct["B"] if "CA" in res] + \
         [res["CA"] for res in exp_struct["A"] if "CA" in res]

af_ca = [af_struct[i]["CA"] for i in range(25, 55)] + \
        [af_struct[i]["CA"] for i in range(90, 111)]

sup = Superimposer()
sup.set_atoms(exp_ca, af_ca)
print(f"Insulin (4INS vs AlphaFold P01308) Cα RMSD = {sup.rms:.2f} Å")

Insulin (4INS vs AlphaFold P01308) Cα RMSD = 7.11 Å


## What this shows about folding
- The amino acid sequence determines the 3D shape.
- Helices and disulfide bonds stabilize insulin's compact core.
- A misfolded protein cannot do its job, which is why folding matters in medicine.